In [875]:
import sys
import os

# This adds the parent directory (..) to the Python path
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

import logging
from datasets import load_dataset
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW

from dataclasses import dataclass, field

from typing import List, Optional, Tuple, Any, Dict

from transformers import (
    PreTrainedModel,
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    TrainerCallback,
    Trainer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_linear_schedule_with_warmup
)

from transformers.modeling_outputs import ModelOutput

DATA_DIR = Path("../data/")
MODELS_DIR = Path("../models/")
pos_list = ['noun',
            'adverb',
            'adjective',
            'verb',
            'preposition',
            'misc',
            'number',
            'not-no',
            'determiner']

### Utils

In [903]:
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import root_mean_squared_error, accuracy_score
from scipy.stats import pearsonr
from wordfreq import zipf_frequency


def merge_cols(batch, cols_to_merge, sep_token):
    """
    Merge specified item text components into a single input string per example.

    Designed for use with Hugging Face `Dataset.map` in batched mode.

    Args:
        batch (dict): Batched dataset examples.
        cols_to_merge (list[str]): Ordered column names to concatenate.
        sep_token (str): Separator string between components.

    Returns:
        dict: Dictionary with key `"input_text"` containing merged text strings.
    """
    rows=zip(*(batch[col] for col in cols_to_merge))
    return {
        "input_text": [
            sep_token.join(str(x).strip() for x in row)
            for row in rows
        ]
    }
    
def preprocess_dataset(ds_dict, cols_to_merge, sep_token):
    """
    Preprocess a Hugging Face DatasetDict by:
        1. Merging specified text columns into a single 'input_text' column.
        2. Renaming the target column 'GLMM_score' to 'label'.
        3. Removing all other columns except 'input_text', 'l1' and 'label'.

    Args:
        ds_dict (datasets.DatasetDict): Input DatasetDict with one or more splits.
        cols_to_merge (list[str]): List of text columns to concatenate into 'input_text'.
        sep_token (str): Separator string to insert between merged columns.

    Returns:
        datasets.DatasetDict: Preprocessed DatasetDict where each split contains only
            the columns 'input_text' and 'label', ready for tokenization.
    """
    
    # Compute columns to remove
    first_split = next(iter(ds_dict.values())) # get first key in ds_dict
    all_columns = first_split.column_names
    cols_to_keep = ["input_text", "L1", "en_target_pos"]
    cols_to_remove = [c for c in all_columns if c not in cols_to_keep and c != 'GLMM_score']
    
    # Format input text, rename target label and remove extra columns
    ds_dict = ds_dict.map(
        merge_cols,
        batched=True,
        fn_kwargs={"cols_to_merge": cols_to_merge, "sep_token": sep_token},
        desc="Formatting input text"
    )
    
    # Compute Zipf frequency of english target word
    ds_dict = ds_dict.map(
        batch_compute_frequency,
        batched=True,
        fn_kwargs={"col":"en_target_word"},
        desc="compute zipf frequency of en target word"
    )
    
    # Rename target columns and remove extra columns
    # final target label lists: ['glmm_label', 'freq_label', 'pos_label']
    ds_dict = ds_dict.rename_column("GLMM_score", "label")
    ds_dict = ds_dict.remove_columns(cols_to_remove)
    
    return ds_dict
    
def batch_compute_frequency(batch, col='en_target_word'):
    """
    zipf_freq: Zipf frequency of a given word based on corpus used by wordfreq
    """
    rows = batch[col]
    return {
            "freq_feature": [zipf_frequency(row.lower(), lang='en') for row in rows]
            }

def cleanup_trainer_memory(*objects):
    """
    Free up memory used by PyTorch and delete given objects.

    Args:
        *objects: Any Python objects (e.g., Trainer, datasets) to delete.
    """
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    for obj in objects:
        del obj

def load_data_paths(data_dir, l1, mode, dataset_split=None):
    """
    Loads CSV file paths for Hugging Face `load_dataset`.

    Returns a dictionary with keys 'train', 'validation', and 'test',
    mapping to lists of CSV paths. Keys with no files are omitted.

    Args:
        data_dir (Path): Base directory containing the data splits.
        l1 (str): Language code (e.g., 'es') or 'xx' to include all languages during finetune.
        mode (str): One of 'finetune', 'predict', or 'evaluate'.
        dataset_split (str or None): For predict/evaluate, which data splits to load:
            'dev', 'test', or 'both'. Ignored for finetune.

    Returns:
        dict: Mapping HF split names to lists of CSV paths, e.g.,
            {'train': ['train/es/file1.csv'], 'validation': ['dev/es/file1.csv']}
    """

    # for finetuning 'xx' means combine all L1s
    # L1s are separate for predict and evaluate
    l1_list = ["es", "de", "cn"] if mode == "finetune" and l1 == "xx" else [l1]

    data_files = {}

    # Mapping from KVL dataset split names to HF split names
    kvl_splits_to_hf = {
        "train": "train",
        "dev": "validation",
        "test": "test"
    }

    if mode == "finetune":
        train_files = []
        val_files = []

        for l1 in l1_list:
            train_folder=data_dir / "train" / l1
            val_folder=data_dir / "dev" / l1

            train_files.extend(str(f) for f in train_folder.glob("*.csv"))
            val_files.extend(str(f) for f in val_folder.glob("*.csv"))

        if train_files:
            data_files["train"] = train_files
        if val_files:
            data_files["validation"] = val_files

    elif mode in {"predict", "evaluate"}:
        
        splits_to_load = ["dev", "test"] if dataset_split == "both" else [dataset_split]

        for folder_name in splits_to_load:
            files = []
            for l1 in l1_list:
                folder = data_dir / folder_name / l1
                files.extend(str(f) for f in folder.glob("*.csv"))

            hf_key = kvl_splits_to_hf[folder_name]
            if files:
                data_files[hf_key] = files

    return data_files

def compute_metrics(eval_pred, debug=False):
    predictions, label_ids = eval_pred
    
    metrics = {}
    
    if debug:
        # --- DEBUGGING PRINTS (Will show up above your progress bar) ---
        print(f"\n[EVAL DEBUG] Predictions type: {type(predictions)}")
        if isinstance(predictions, tuple):
            print(f"[EVAL DEBUG] Predictions tuple length: {len(predictions)}")
            
        print(f"[EVAL DEBUG] Label IDs type: {type(label_ids)}")
        if isinstance(label_ids, tuple):
            print(f"[EVAL DEBUG] Label IDs tuple length: {len(label_ids)}")
        # -------------------------------------------------------------

    # 1. Primary Task (Difficulty)
    logits = predictions[0] if isinstance(predictions, tuple) else predictions
    labels = label_ids[0] if isinstance(label_ids, tuple) else label_ids
    
    logits = logits.flatten()
    labels = labels.flatten()
    
    metrics['rmse'] = root_mean_squared_error(labels, logits)
    metrics['pearson'] = pearsonr(logits, labels)[0]
    
    # 2. Auxiliary Task (POS tagging)
    # If the Trainer correctly passed both outputs and both labels...
    if isinstance(predictions, tuple) and len(predictions) > 1:
        if isinstance(label_ids, tuple) and len(label_ids) > 1:
            
            pos_logits = predictions[1]
            pos_labels = label_ids[1]
            
            # Ensure they aren't None before calculating
            if pos_logits is not None and pos_labels is not None:
                pos_logits = pos_logits
                pos_labels = pos_labels.flatten()
                pos_preds = np.argmax(pos_logits, axis=-1)
                metrics["pos_acc"] = accuracy_score(pos_labels, pos_preds)
            else:
                print("[EVAL DEBUG] Freq Logits or Freq Labels are None!")
        else:
            print("[EVAL DEBUG] label_ids is not a tuple. Did you set label_names?")
            
    return metrics

In [877]:
### analyzing
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def analyze_predictions(trainer, dataset, data_collator, tokenizer, pos_list):
    """
    Runs inference on a dataset and extracts the isolated Baseline Difficulty 
    and Context Shift for explicit psychometric analysis.
    """
    model = trainer.model
    model.eval() # Freeze dropout and batch norm
    
    # Create a PyTorch DataLoader
    dataloader = DataLoader(
        dataset, 
        batch_size=32, 
        collate_fn=data_collator,
        shuffle=False
    )
    
    results = []
    
    # Reverse mapping for POS tags (Idx -> String)
    idx_to_pos = {i: pos for i, pos in enumerate(pos_list)}

    print("Running deep inference and extracting feature splits...")
    with torch.no_grad():
        for batch in tqdm(dataloader):
            # Move batch to the same device as the model (GPU)
            batch = {k: v.to(model.device) for k, v in batch.items()}
            
            # 1. Forward Pass
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                freq_feature=batch.get("freq_feature")
            )
            
            # 2. Extract Total Difficulty and POS Predictions
            total_pred_diff = outputs.logits.view(-1).cpu().numpy()
            pos_logits = outputs.pos_logits.cpu().numpy()
            predicted_pos_idx = np.argmax(pos_logits, axis=-1)
            
            # 3. Mathematically Isolate the Baseline and Context Shift
            # We explicitly pass the frequency feature through the linear prior layer
            freq_feature = batch["freq_feature"].view(-1, 1)
            baseline_diff = model.freq_to_baseline(freq_feature).view(-1).cpu().numpy()
            
            # Since Total = Baseline + Context Shift, then Context Shift = Total - Baseline
            context_shift = total_pred_diff - baseline_diff
            
            # 4. Decode text and gather true labels
            input_texts = tokenizer.batch_decode(batch["input_ids"], skip_special_tokens=True)
            true_diff = batch["label"].cpu().numpy() if "label" in batch else [None] * len(input_texts)
            raw_freqs = batch["freq_feature"].cpu().numpy()
            
            true_pos = batch["pos_label"].cpu().numpy() if "pos_label" in batch else [None] * len(input_texts)

            # 5. Store row by row
            for i in range(len(input_texts)):
                results.append({
                    "Input Text": input_texts[i],
                    "Z-Freq": round(raw_freqs[i], 3),
                    "Baseline (Freq)": round(baseline_diff[i], 4),
                    "Context Shift (XLMR)": round(context_shift[i], 4),
                    "Total Predicted Difficulty": round(total_pred_diff[i], 4),
                    "True GLMM Difficulty": round(true_diff[i], 4) if true_diff[i] is not None else None,
                    "Pred POS": idx_to_pos.get(predicted_pos_idx[i], "Unknown"),
                    "True POS": idx_to_pos.get(true_pos[i], "Unknown") if true_pos[i] is not None else None,
                })
                
    # Convert to Pandas DataFrame
    df = pd.DataFrame(results)
    
    # Add an Absolute Error column for the GLMM task
    if "True GLMM Difficulty" in df.columns and df["True GLMM Difficulty"].notna().any():
        df["Abs Error"] = abs(df["Total Predicted Difficulty"] - df["True GLMM Difficulty"])
        
    return df

### Output class and Collater

In [878]:
class OptimizerStateMonitor(TrainerCallback):
    """
    A custom Hugging Face callback to inspect optimizer groups, learning rates, 
    and the real-time values/gradients of specific custom parameters.
    """
    def __init__(self, params_to_watch: list):
        # Pass in strings that match the parameter names you want to track
        self.params_to_watch = params_to_watch

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Triggers exactly when the Trainer prints the training loss to the console."""
        
        optimizer = kwargs.get("optimizer")
        model = kwargs.get("model")

        print(f"\n{'-'*60}")
        print(f"🔍 OPTIMIZER & PARAMETER STATE | Step: {state.global_step}")
        print(f"{'-'*60}")
        
        # Map Parameter Memory IDs to their Optimizer Group Index
        param_to_group_idx = {}

        # 1. Inspect Optimizer Parameter Groups
        if optimizer is not None:
            print("Optimizers (Learning Rates & Weight Decay):")
            for i, group in enumerate(optimizer.param_groups):
                lr = group.get('lr', 0.0)
                weight_decay = group.get('weight_decay', 0.0)
                # Count how many parameter tensors are in this group
                num_params = len(group.get('params', []))
                print(f"  [Group {i}] Params: {num_params} | LR: {lr:.2e} | L2 Decay: {weight_decay}")
                
                # Store the group index for every parameter object in this group
                for p in group.get('params', []):
                    param_to_group_idx[id(p)] = i
        else:
            print("  [Warning] Optimizer not found in kwargs.")

        # 2. Inspect Custom Parameter Values and Live Gradients
        if model is not None:
            
            print("\nTracked Parameters (Values & Gradients):")
            for name, param in model.named_parameters():
                # Check if this parameter name contains any of our target strings
                if any(target in name for target in self.params_to_watch):
                    
                    # Safely detach and convert to numpy for clean printing
                    val = param.data.detach().cpu().numpy()
                    
                    # Check if gradients have been computed yet
                    if param.grad is not None:
                        grad = param.grad.data.detach().cpu().numpy()
                    else:
                        grad = "No Gradient Yet"
                    
                    # Look up the group using the parameter's unique memory ID
                    group_idx = param_to_group_idx.get(id(param), "Unassigned")

                    # Format the output cleanly
                    print(f"  -> {name} [Group {group_idx}]:")
                    print(f"       Value : {np.array2string(val, precision=6, separator=', ')}")
                    print(f"       Grad  : {np.array2string(grad, precision=6, separator=', ') if isinstance(grad, np.ndarray) else grad}")
        print(f"{'-'*60}\n")

In [ ]:
@dataclass
class CustomClassifierOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None

@dataclass
class CustomDataCollator(DataCollatorWithPadding):
    custom_features: List[str] = field(default_factory=lambda: ["freq_feature", "pos_label", "label"])

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        extracted_features = {key: [] for key in self.custom_features}
        
        for feature in features:
            for key in self.custom_features:
                if key in feature:
                    extracted_features[key].append(feature.pop(key))

        batch = super().__call__(features)

        if "pos_label" in extracted_features and extracted_features["pos_label"]:
            batch["pos_label"] = torch.tensor(extracted_features["pos_label"], dtype=torch.long)
        
        if "freq_feature" in extracted_features and extracted_features['freq_feature']:
            batch['freq_feature'] = torch.tensor(extracted_features['freq_feature'], dtype=torch.float32)
        
        if "label" in extracted_features and extracted_features["label"]:
            batch["label"] = torch.tensor(extracted_features["label"], dtype=torch.float32)
            
        return batch

### Layer-wise pooling module

In [901]:
class Rational(nn.Module):
    """
    Rational Activation Function (Version A)
    Translated natively for PyTorch 2.1.0+
    
    Formula: F(x) = P(x) / (1 + |Q(x)|)
    P(x) = a_0 + a_1*x + a_2*x^2 + ... + a_m*x^m
    Q(x) =       b_1*x + b_2*x^2 + ... + b_n*x^n
    """
    def __init__(self, approx_func="leaky_relu", degrees=(5, 4), trainable=True):
        super(Rational, self).__init__()
        self.degrees = degrees
        
        # The official library initializes the weights to approximate standard functions.
        # Here are the exact pre-computed coefficients to approximate Leaky ReLU.
        if approx_func == "leaky_relu" and degrees == (5, 4):
            init_num = [0.0274, 0.4907, 0.0538, 0.1655, 0.0272, 0.0381]
            init_den = [         0.9329, 0.0210, 0.3341, 0.0221]
        elif approx_func == "identity":
            # Fallback: F(x) = x
            init_num = [0.0] * (degrees[0] + 1)
            init_num[1] = 1.0
            init_den = [0.0] * degrees[1]
        else:
            raise ValueError("Only 'leaky_relu' or 'identity' initializations are provided in this script.")

        # Register parameters
        self.numerator = nn.Parameter(torch.tensor(init_num, dtype=torch.float32), requires_grad=trainable)
        self.denominator = nn.Parameter(torch.tensor(init_den, dtype=torch.float32), requires_grad=trainable)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Evaluate Numerator P(x) using Horner's Method
        num = self.numerator[-1]
        for i in range(self.degrees[0] - 1, -1, -1):
            num = self.numerator[i] + x * num
            
        # 2. Evaluate Denominator Q(x) using Horner's Method
        den = self.denominator[-1]
        for i in range(self.degrees[1] - 2, -1, -1):
            den = self.denominator[i] + x * den
        
        # Multiply by x one last time because Q(x) has no b_0 constant term
        den = x * den 
        
        # 3. Apply Version A Bounds (1 + |Q(x)|)
        den = 1.0 + torch.abs(den)
        
        # 4. Final Rational Output
        return num / den

In [900]:
class ScalarMix(nn.Module):
    def __init__(
        self, 
        mixture_size: int = 13, 
        temperature: float = 2.0,       # Enforces a smooth distribution
        layer_dropout: float = 0.1      # Forces the network to use multiple layers
    ) -> None:
        """
        A stabilized Layer Mixer designed for Multi-Task architectures.
        Prevents Softmax Saturation and Pre-Norm variance explosions.
        """
        super().__init__()
        self.mixture_size = mixture_size
        self.temperature = temperature
        # Initialize uniformly. With temperature, this starts as a perfectly flat distribution.
        self.scalar_weights = nn.Parameter(torch.zeros(mixture_size))
        self.gamma = nn.Parameter(torch.tensor([1.0]))
        # self.register_buffer('gamma', torch.tensor([1.0]))

    def forward(
        self, 
        tensors: List[torch.Tensor],
        mask: torch.bool = None
    ) -> torch.Tensor:
        
        # Locks raw weights between -1.0 and 1.0. Impossible to explode.
        # bounded_weights = torch.tanh(self.scalar_weights)
        
        # 1. Calculate softmax(w_j)
        normed_weights = F.softmax(self.scalar_weights, dim=0)

        # 2. Stack the transformer layer outputs
        # Shape: (num_layers, batch_size, seq_len, hidden_size)
        tensors_stacked = torch.stack(tensors, dim=0)
        
        # 3. Broadcast weights to match the tensor dimensions
        weights_shape = (-1,) + (1,) * (tensors_stacked.dim() - 1)
        normed_weights = normed_weights.view(weights_shape)
        
        # 4. Calculate sum(softmax(w_j) * t_j)
        mixed_tensor = torch.sum(normed_weights * tensors_stacked, dim=0)
        
        # mixed_tensor = torch.clamp(mixed_tensor, min = 1e-9, max = 10)
        
        # 5. Apply gamma multiplier
        # Optional: Mask out padding tokens for the downstream pooler
        if mask is not None:
            broadcast_mask = mask.unsqueeze(-1)
            mixed_tensor = mixed_tensor * broadcast_mask


        return self.gamma * mixed_tensor

### Token-wise pooling module

In [883]:
class PositionalEncoding(nn.Module): 
    """Positional encoding."""
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # Create a long enough P
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)
        
class LearnedPositionalEncoding(nn.Module):
    def __init__(self, max_seq_len=128, latent_dim=768):
        super().__init__()
        self.positional_embedding = nn.Embedding(max_seq_len, latent_dim)
    
    def forward(self, x):
        positions = torch.arange(x.size(-2)).expand(x.size(0), -1).to(x.device)
        embed = self.positional_embedding(positions)
        return x + embed

class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size, dropout):
        super().__init__()
        # Projects hidden states to compute attention scores
        self.W = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden_states, attention_mask):
        """
        hidden_states: [batch_size, seq_len, hidden_size]
        attention_mask: [batch_size, seq_len]
        """
        # 1. Compute unnormalized attention scores
        # Shape: [batch_size, seq_len, 1] -> [batch_size, seq_len]
        scores = self.v(torch.tanh(self.W(hidden_states))).squeeze(-1)
        
        # 2. Mask out padding tokens so they don't receive attention
        # We use a large negative number (-1e9) so softmax outputs 0
        scores = scores.masked_fill(attention_mask == 0, -1e9)
        
        # 3. Normalize scores into probabilities (alpha weights)
        attn_weights = torch.softmax(scores, dim=-1) # [batch_size, seq_len]
        
        # 4. Compute the weighted sum of hidden states (Context Vector)
        # unsqueeze attn_weights to [batch, 1, seq_len] for batch matrix multiplication
        # output shape: [batch, 1, hidden_size] -> [batch, 1, hidden_size]
        context_vector = torch.bmm(self.dropout(attn_weights).unsqueeze(1), hidden_states)
        
        return context_vector, attn_weights

### Model

In [905]:
class Backbone(PreTrainedModel):
    config_class = AutoConfig
    def __init__(
        self,
        config
    ):
        """
        Instantiate the model backbone as a subclass of PreTrainedModel.
        Due to transformer's design pattern, we have to assign model prefix for backbones of different structures. 
        e.g. roberta: ["xlm-roberta", "roberta"]
        """
        super().__init__(config)
        self.config = config
        model_type = config.model_type
        if model_type in ['xlm-roberta', 'roberta']:
            # add_pooling_layer=False to get raw hidden states
            self.roberta = AutoModel.from_config(config, add_pooling_layer=False)
            self.__class__.base_model_prefix = "roberta"
    
    def forward(self):
        raise NotImplementedError("ERROR: forward not implemented!")

class CloseTrack_Predictor(Backbone):
    def __init__(
        self,
        config,
        dropout: float = 0.1,
        layer_wise: Optional[str] = None,
        token_agg: Optional[str] = None,
        pred_head: str = 'mlp',
        temp: float = 2.0,
        **kwargs
    ):
        """
        Our customized predictor that performs layer fusion and/or token fusion strategies to exploit transformer's internal structure. Currently, we have implemented 
        one method for layer fusion (layer_wise) and two for token fusion (token_wise). If both are not provided, the predictor reduces to the ordinary regressor, 
        using the first token from the last hidden state as predictor. 
            - layer_wise:
                - "ScalarMix": the layer fusion method that learns a global set of weights for each internal layers before pooling into a single hidden state. 
            - token_wise: 
                - 'AddAttn': the Additive Attention (see [https://d2l.ai/chapter_attention-mechanisms-and-transformers/attention-scoring-functions.html#additive-attention])
                - 'SelfAttn': the Multi-head self-attention, implemented by the PyTorch off-the-self nn.MultiheadAttention function.
                    Since the Positional Encoding are used in transformer training, two types of positional encoder are used for self-attention for investigation
                    - Sinusodial: [https://d2l.ai/chapter_attention-mechanisms-and-transformers/self-attention-and-positional-encoding.html#positional-encoding]
                    - Learned: [https://machinelearningmastery.com/positional-encodings-in-transformer-models/]
            - pred_head: the regressor used for prediction
                - 'mlp': the same MLP regressor as used in XLMRobertaForSequenceClassification
                - 'vib': a testing regressor that follows the idea of Variational Information Bottleneck, which is essentially a regularized regressor.
        """
        super().__init__(config)

        # initiate layer fusion strategy
        self.layer_wise = layer_wise
        if self.layer_wise == 'ScalarMix':
            ScalarMix(
                config.num_hidden_layers + 1,
                temperature=temp,
                layer_dropout=dropout
                )

        self.token_agg == token_agg

        # initiate regressor
        self.pred_head = pred_head
        if self.pred_head == 'mlp':
            self.regressor = nn.Sequential(
                nn.Linear(config.hidden_size, config.hidden_size),
                Rational(approx_func="leaky_relu", degrees=(5, 4)),
                nn.Dropout(dropout),
                nn.Linear(config.hidden_size, config.num_label)
            )

        # Initialize weights (Hugging Face standard)
        self.post_init()

    def forward(
        self,
        input_ids,
        attention_mask,
        label=None,        
        freq_feature=None,
        pos_label=None,
        **kwargs
        ) -> CustomClassifierOutput:
        """
        Argument:
            input_ids: the tokenized input 
            attention_mask: the sequence mask output by the tokenizer
            labels: the target labels
        Return:
            CustomClassifierOutput(
                loss: prediction loss
                logits: predicted labels
                hidden_states: all hidden states generated by the backbone
                attentions: all attention scores calculated by the backbone
                token_attn_weights: the token-level attention weights calculated by the token fusion method.
            )
        """

        # The Trainer passes 'num_items_in_batch' which the backbone doesn't accept.
        if "num_items_in_batch" not in self.config:
            kwargs.pop("num_items_in_batch", None)

        # identify the backbone and generate the hidden states. 
        if hasattr(self, "roberta"):
            outputs = self.roberta(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                **kwargs
            )

        # if layer fusion is applied
        if self.layer_wise:
            # mixes the entire sequences from all layers, not just the first token to support following token-wise aggregation
            hiddens = list(outputs.hidden_states) # [layer, batch, seq, hidden]

            if self.layer_wise == "ScalarMix":          
                    hiddens = self.layer_agg(hiddens, mask = attention_mask) # [1, batch, seq, hidden]
                    
        else:
            # else, uses only the last hidden states.
            hiddens = outputs.last_hidden_state # [batch, seq, hidden]
   
        # # predict with the first token
        if self.token_agg == 'sentence':
            hiddens = hiddens[:, 0, :] # [batch, 1, hidden]
        if self.token_agg == 'mean_token':
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(-1, -1, hiddens.size(-1)).float()
            
            # 1. Multiply by the mask to strictly zero-out all [PAD] tokens
            masked_embeddings = hiddens * input_mask_expanded
            
            # 2. Sum the valid embeddings along the sequence length (dim=1)
            sum_embeddings = torch.sum(masked_embeddings, dim=1)
            
            # 3. Sum the mask to find the true, unpadded length of each sequence
            # We use torch.clamp to mathematically prevent division by zero
            sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
            
            # 4. Divide the sum by the true sequence length to get the precise mean
            hiddens = sum_embeddings / sum_mask 
                
            # Final shape of hiddens: [batch_size, hidden_size]
        
            loss = None
            logits = self.regressor(hiddens)
        
            if label is not None:
                loss_fct_mse = nn.MSELoss(reduction='mean')
                loss_fct_ce = nn.CrossEntropyLoss(reduction='mean')
               
                loss = loss_fct_mse(logits.view(-1), label.view(-1))
        
        return CustomClassifierOutput(
            loss=loss,
            logits=logits,
            attentions=outputs.attentions
        )

## Train

In [885]:
@dataclass
class ModelParameters:
    # data params
    pre_trained_model: Optional[str] = None
    l1: Optional[str] = None
    component_order: Optional[str] = None
    # model params
    num_class_list: List[int] = field(default_factory=lambda: [1])
    cls_only: bool = False
    num_attn_heads: Optional[int] = None
    last_k_layer: Optional[int] = None
    target_list: List[str] = None
    loss_weight: List[float] = field(default_factory=lambda: [1.0])
    dynamic_weight: bool = False
    temp: Optional[float] = None
    beta: Optional[float] = None
    z_dim: Optional[int] = None
    dropout: Optional[float] = None
    pred_head: Optional[str] = None
    layer_wise: Optional[str] = None
    token_wise: Optional[str] = None
    pos_encoding: Optional[str] = None
    learned_pos: Optional[bool] = None
    max_seq_len: Optional[int] = None
    # train params
    epoch: Optional[int] = None
    batch_size: Optional[int] = None
    learning_rate: Optional[float] = None
    weight_decay: Optional[float] = None
    warmup_ratio: Optional[float] = None
    

In [ ]:
def run_finetune(model_params, seed):
    """
    Fine-tune pre-trained transformer models.
    """
    
    # Checking cuda availability
    if torch.cuda.is_available():
        logging.info(f"Training on GPU: {torch.cuda.get_device_name(0)}")
    else:
        logging.warning("Training on CPU.")

    backbone = model_params.pre_trained_model 
    l1 = model_params.l1
    beta = model_params.beta
    z_dim = model_params.z_dim
    epoch = model_params.epoch
    last_k_layer = model_params.last_k_layer
    cls_only = model_params.cls_only
    num_heads = model_params.num_attn_heads
    target_list = model_params.target_list
    num_class_list = model_params.num_class_list
    temp = model_params.temp 
    dynamic_weight = model_params.dynamic_weight
    loss_weight = model_params.loss_weight
    batch_size = model_params.batch_size
    learning_rate = model_params.learning_rate
    weight_decay = model_params.weight_decay
    warmup_ratio = model_params.warmup_ratio
    dropout = model_params.dropout
    pred_head = model_params.pred_head
    layer_wise = model_params.layer_wise
    token_wise = model_params.token_wise
    pos_encoding = model_params.pos_encoding
    learned_pos = model_params.learned_pos
    max_seq_len = model_params.max_seq_len
    component_order = model_params.component_order

    # Only cast if they exist, otherwise leave as None
    if beta is not None:
        beta = float(beta)
    if z_dim is not None:
        z_dim = int(z_dim)
    if dropout is not None:
        dropout = float(dropout)
    if max_seq_len is not None:
        max_seq_len = int(max_seq_len)
        
    logging.info(f"""
        ---------------------Model Setup---------------------
        Using regression head: {pred_head}
        Layer-wise aggregation: {layer_wise if layer_wise else False}
        Token-wise aggregation: {token_wise if token_wise else False}
        Positional Encoding: {bool(int(pos_encoding)) if pos_encoding else False}
        Learned Positional Encoder: {bool(int(learned_pos)) if learned_pos else False}
        Dropout: {dropout}
        -----------------------------------------------------""")

    if pos_encoding:
        logging.info(f"Maximum Sequence Length for Positional Encoder: {max_seq_len}")
                
    def model_init_function():
        return CloseTrack_Predictor.from_pretrained(
            backbone, 
            num_labels=1,
            temp = temp,
            dropout=dropout,
            layer_wise=layer_wise,
            token_agg=token_agg
        )

    try:
        logging.info(f"Fine-tuning model...")

        # Load dataset paths and Hugging Face DatasetDict
        data_files = load_data_paths(DATA_DIR, l1, "finetune")
        hf_dataset = load_dataset("csv", data_files=data_files)

        # Load tokenizer and prepare input text formatting
        tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=True)
        cols_to_merge = component_order.split("; ")
        sep_token = f" {tokenizer.sep_token} " if tokenizer.sep_token else " "

        # Preprocess dataset: format input text, rename target label and remove extra columns
        preprocessed_ds = preprocess_dataset(hf_dataset, cols_to_merge, sep_token)

        # Tokenize dataset
        tokenized_ds = preprocessed_ds.map(
            lambda x: tokenizer(x["input_text"], truncation=True),
            batched=True,
            desc="Tokenizing input text"
        )

        # Itemize pos tagging 
        pos_to_idx = {pos:i for i, pos in enumerate(pos_list)}
        tokenized_ds = tokenized_ds.map(
            lambda x: {"pos_label": [pos_to_idx[val] for val in x['en_target_pos']]},
            batched=True,
            remove_columns=['en_target_pos', 'L1', 'input_text'],
            desc="Itemizing POS tagging"
        )

        # Set up training arguments
        training_args = TrainingArguments(
            # output_dir=str(MODELS_DIR / model_name),
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_strategy="epoch",
            # save_total_limit=1,
            # max_grad_norm=1.0, # gradient clipping
            num_train_epochs=int(epoch),
            per_device_train_batch_size=int(batch_size),
            per_device_eval_batch_size=int(batch_size),
            learning_rate=float(learning_rate),
            weight_decay=float(weight_decay),
            warmup_ratio=float(warmup_ratio),
            load_best_model_at_end=True,
            report_to="none",
            seed=seed,
            label_names=target_list
        )

        # Initialise trainer
        data_collator = CustomDataCollator(tokenizer, custom_features=["freq_feature", "pos_label", "label"])

        # set customized optimizer
        # 1. Initialize the model first so we can access its parameters
        model = model_init_function()
        custom_param_names = ["numerator", "denominator"] # [] 
        
        custom_params = [p for n, p in model.named_parameters() if any(nd in n for nd in custom_param_names)]
        base_params = [p for n, p in model.named_parameters() if not any(nd in n for nd in custom_param_names)]

        # 3. Create Custom Optimizer with separate rules
        optimizer = AdamW([
            {
                "params": base_params, 
                "lr": training_args.learning_rate, 
                "weight_decay": training_args.weight_decay
            },
            {
                "params": custom_params, 
                "lr": 1e-2,             # Massive LR for the scalars
                "weight_decay": 0.0     # STRICTLY ZERO weight decay
            }
        ])

        # 4. Rebuild the learning rate scheduler (Trainer usually does this automatically, 
        # but we must do it manually since we are overriding the optimizer)
        num_training_steps = (len(tokenized_ds["train"]) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
        num_warmup_steps = int(num_training_steps * training_args.warmup_ratio)
        
        lr_scheduler = get_linear_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps
        )
        
        # Initialize the callback to watch your specific custom architecture weights
        debug_callback = OptimizerStateMonitor(
            params_to_watch=["weight_logvars", "freq_to_baseline", "scalar_weights", 'gamma', 'layer_weights'] 
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_ds["train"],
            eval_dataset=tokenized_ds["validation"],
            data_collator=data_collator,
            optimizers=(optimizer, lr_scheduler),
            compute_metrics=compute_metrics,
            callbacks=[debug_callback]
        )
        
        # trainer = Trainer(
        #     model_init=model_init_function,
        #     args=training_args,
        #     train_dataset=tokenized_ds["train"],
        #     eval_dataset=tokenized_ds["validation"],
        #     data_collator=data_collator,
        #     compute_metrics=compute_metrics
        # )

        # Verify model is on the correct device
        logging.info(f"Model successfully loaded on: {trainer.model.device}")

        # Finetune model and save
        trainer.train()
        logging.info(f"Model fine-tuned ")
        # analysis_df = analyze_predictions(
        #     trainer=trainer, 
        #     dataset=tokenized_ds["validation"], 
        #     data_collator=data_collator, 
        #     tokenizer=tokenizer,
        #     pos_list=pos_list
        # )
        
    except Exception:
        logging.exception(f"Failed model")
        raise
    
    return trainer

In [892]:
mlp_mtl_ScalarMix_params = ModelParameters(
    pre_trained_model = 'xlm-roberta-base',
    l1 = 'xx',
    component_order = "L1_source_word; L1_context; en_target_clue; en_target_word",
    # model params
    cls_only = False, 
    num_attn_heads = 4,
    last_k_layer = 3,
    num_class_list=[1],
    target_list = ['label'],
    loss_weight = [],
    dynamic_weight=True,
    temp = 10000,
    beta = 1e-3,
    z_dim = 31,
    dropout = 0.4,
    pred_head = 'mlp',
    layer_wise = 'ScalarMix',
    token_wise = None,
    pos_encoding = None,
    learned_pos = None,
    max_seq_len = None,
    # train params
    epoch = 5,
    batch_size = 64,
    learning_rate = 5e-3,
    weight_decay = 0.1,
    warmup_ratio = 0.1
)

In [893]:
trainer = run_finetune(mlp_mtl_ScalarMix_params, 1234)

Some weights of CloseTrack_Predictor were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['layer_agg.0.gamma', 'layer_agg.0.scalar_weights', 'regressor.0.0.bias', 'regressor.0.0.weight', 'regressor.0.1.denominator', 'regressor.0.1.numerator', 'regressor.0.3.bias', 'regressor.0.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Rmse,Pearson
1,744110750026225456995368960.000000,376164381837606690112929792.000000,19394956623872.000000,-0.269077
2,180570712335494874417594368.000000,78914138129661334078357504.000000,8883362856960.000000,-0.230605
3,44436374698979971316056064.000000,26298294011563796840054784.000000,5128186232832.000000,-0.220264
4,17840861151603192280121344.000000,13816359974320449867743232.000000,3717036507136.000000,-0.230890
5,11433121699247576949194752.000000,11234519086118922818158592.000000,3351793369088.000000,-0.236559



------------------------------------------------------------
🔍 OPTIMIZER & PARAMETER STATE | Step: 286
------------------------------------------------------------
Optimizers (Learning Rates & Weight Decay):
  [Group 0] Params: 203 | LR: 4.44e-03 | L2 Decay: 0.1
  [Group 1] Params: 2 | LR: 8.88e-03 | L2 Decay: 0.0

Tracked Parameters (Values & Gradients):
  -> layer_agg.0.scalar_weights [Group 0]:
       Value : [3.200421e+06, 2.786202e-41, 3.200458e+06, 2.786202e-41, 3.200458e+06,
 2.786202e-41, 0.000000e+00, 0.000000e+00, 0.000000e+00, 0.000000e+00,
 0.000000e+00, 0.000000e+00, 0.000000e+00]
       Grad  : No Gradient Yet
  -> layer_agg.0.gamma [Group 0]:
       Value : [0.002227]
       Grad  : No Gradient Yet
------------------------------------------------------------

[EVAL DEBUG] label_ids is not a tuple. Did you set label_names?

------------------------------------------------------------
🔍 OPTIMIZER & PARAMETER STATE | Step: 286
----------------------------------------------

In [19]:
cleanup_trainer_memory(trainer)

NameError: name 'trainer' is not defined

In [816]:
model = trainer.model

In [429]:
torch.exp(model.weight_logvars.data)

tensor([1.3790, 1.1848], device='cuda:0')

In [761]:
model.layer_agg[0].layer_weights.data

tensor([-0.0014, -0.0038, -0.0044, -0.0045, -0.0045, -0.0045, -0.0046, -0.0019,
         0.0007,  0.0011,  0.0010,  0.0011,  0.0032], device='cuda:0')

In [817]:
F.softmax(model.layer_agg[0].scalar_weights.data, 0)

tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.], device='cuda:0')

In [430]:
[(object, F.softmax(i.scalar_weights.data, 0)) for object, i in zip(["Primary", "POS"], model.layer_agg)]

[('Primary',
  tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.], device='cuda:0')),
 ('POS',
  tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0'))]

In [42]:
for name, params in trainer.model.layer_agg.named_parameters():
    print(f"{name}: {params}")

0.scalar_weights: Parameter containing:
tensor([2.4898e+00, 7.5782e-42, 2.4898e+00, 7.5782e-42, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 4.6606e-08, 4.5912e-41, 4.6606e-08, 4.5912e-41,
        5.6052e-45, 0.0000e+00, 3.0846e+06, 0.0000e+00, 4.6606e-08, 4.5912e-41,
        4.6606e-08, 4.5912e-41, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00], device='cuda:0', requires_grad=True)


In [46]:
F.softmax(trainer.model.layer_agg[0].scalar_weights)

/scratch/local/27751392/ipykernel_28808/2197195660.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  F.softmax(trainer.model.layer_agg[0].scalar_weights)


tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)

In [238]:
analysis

,Input Text,Z-Freq,Baseline (Freq),Context Shift (XLMR),Total Predicted Difficulty,True GLMM Difficulty,Pred POS,True POS,Abs Error
0,extraña Esta situación es muy extraña y no ent...,4.02,0.9718,-1.6623,-0.6906,-0.9001,adjective,adjective,0.2095
1,evidencia ¿Nos puede enseñar evidencia de su i...,4.70,1.1731,-2.4502,-1.2771,-1.0550,noun,noun,0.2221
2,nuevo El descubrimiento nuevo de los restos de...,5.04,1.2738,-2.5258,-1.2520,-0.5173,adjective,adjective,0.7347
3,ayudante Trabaja como ayudante al Doctor Gonzá...,4.72,1.1790,-1.5085,-0.3295,-0.1453,noun,noun,0.1842
4,primavera La primavera es mi estación preferid...,4.92,1.2383,1.6069,2.8451,2.6686,noun,noun,0.1765
...,...,...,...,...,...,...,...,...,...
2026,"分开地(表示与其他事物有距离),在一边 1)他两脚分开地站着。2)距离不能让我们分开。 a_...",4.78,1.1968,0.0712,1.2680,0.7024,adverb,adverb,0.5656
2027,"图画,照片 1)她的书桌上摆了一张与丈夫的照片。2)童书中有许多彩图。 p______ pi...",5.13,1.3004,0.8171,2.1175,1.5418,noun,noun,0.5757
2028,"(使)倾斜、翻倒 1)船向一边倾斜。2)我们撬开箱子,把箱里的茶叶都倾倒进海里。 t__ tip",4.55,1.1287,-2.0021,-0.8734,-2.6769,verb,verb,1.8035
2029,最近的 最近的医院在哪? n______ nearest,4.11,0.9984,-0.5165,0.4819,0.6236,adjective,adjective,0.1417


In [225]:
analysis['Baseline (Freq)']

0      -0.0060
1      -0.0070
2      -0.0075
3      -0.0071
4      -0.0074
         ...  
2026   -0.0072
2027   -0.0077
2028   -0.0068
2029   -0.0062
2030   -0.0072
Name: Baseline (Freq), Length: 2031, dtype: float32

In [463]:
model_params = ModelParameters(
    pre_trained_model = 'xlm-roberta-base',
    l1 = 'xx',
    component_order = "L1_source_word; L1_context; en_target_clue; en_target_word",
    # model params
    cls_only = False, 
    num_attn_heads = 4,
    last_k_layer = 3,
    num_class_list=[1,len(pos_list)],
    target_list = ['label', "pos_label"],
    loss_weight = [1],
    dynamic_weight=True,
    beta = 1e-3,
    z_dim = 31,
    dropout = 0.1,
    pred_head = 'mlp',
    layer_wise = 'ScalarMix',
    token_wise = None,
    pos_encoding = None,
    learned_pos = None,
    max_seq_len = None,
    # train params
    epoch = 5,
    batch_size = 64,
    learning_rate = 3e-5,
    weight_decay = 0.1,
    warmup_ratio = 0.1
)
backbone = model_params.pre_trained_model
l1 = model_params.l1
beta = model_params.beta
z_dim = model_params.z_dim
epoch = model_params.epoch
last_k_layer = model_params.last_k_layer
cls_only = model_params.cls_only
num_heads = model_params.num_attn_heads
target_list = model_params.target_list
num_class_list = model_params.num_class_list
dynamic_weight = model_params.dynamic_weight
loss_weight = model_params.loss_weight
batch_size = model_params.batch_size
learning_rate = model_params.learning_rate
weight_decay = model_params.weight_decay
warmup_ratio = model_params.warmup_ratio
dropout = model_params.dropout
pred_head = model_params.pred_head
layer_wise = model_params.layer_wise
token_wise = model_params.token_wise
pos_encoding = model_params.pos_encoding
learned_pos = model_params.learned_pos
max_seq_len = model_params.max_seq_len
component_order = model_params.component_order
model_ini = CloseTrack_Predictor.from_pretrained(
            backbone, 
            num_labels=1,
            last_k_layer = last_k_layer,
            cls_only = cls_only,
            target = target_list,
            num_class_list = num_class_list,
            loss_weight = loss_weight,
            dynamic_weight = dynamic_weight,
            num_heads = num_heads,
            dropout=dropout,
            latent_dim=z_dim,
            beta=beta,
            layer_wise=layer_wise,
            token_wise=token_wise,
            pred_head=pred_head,
            pos_encoding=pos_encoding,
            learned_pos=learned_pos,
            max_seq_len=max_seq_len
        )

Some weights of CloseTrack_Predictor were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['freq_to_baseline.bias', 'freq_to_baseline.weight', 'layer_agg.0.gamma', 'layer_agg.0.scalar_weights', 'layer_agg.1.gamma', 'layer_agg.1.scalar_weights', 'regressor.0.0.bias', 'regressor.0.0.weight', 'regressor.0.1.den_coeffs', 'regressor.0.1.num_coeffs', 'regressor.0.3.bias', 'regressor.0.3.weight', 'regressor.1.0.bias', 'regressor.1.0.weight', 'regressor.1.1.den_coeffs', 'regressor.1.1.num_coeffs', 'regressor.1.3.bias', 'regressor.1.3.weight', 'weight_logvars']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [464]:
for n, p in model_ini.named_parameters():
    print(n)

weight_logvars
roberta.embeddings.word_embeddings.weight
roberta.embeddings.position_embeddings.weight
roberta.embeddings.token_type_embeddings.weight
roberta.embeddings.LayerNorm.weight
roberta.embeddings.LayerNorm.bias
roberta.encoder.layer.0.attention.self.query.weight
roberta.encoder.layer.0.attention.self.query.bias
roberta.encoder.layer.0.attention.self.key.weight
roberta.encoder.layer.0.attention.self.key.bias
roberta.encoder.layer.0.attention.self.value.weight
roberta.encoder.layer.0.attention.self.value.bias
roberta.encoder.layer.0.attention.output.dense.weight
roberta.encoder.layer.0.attention.output.dense.bias
roberta.encoder.layer.0.attention.output.LayerNorm.weight
roberta.encoder.layer.0.attention.output.LayerNorm.bias
roberta.encoder.layer.0.intermediate.dense.weight
roberta.encoder.layer.0.intermediate.dense.bias
roberta.encoder.layer.0.output.dense.weight
roberta.encoder.layer.0.output.dense.bias
roberta.encoder.layer.0.output.LayerNorm.weight
roberta.encoder.layer.0.o

In [362]:
data_files = load_data_paths(DATA_DIR, l1, "finetune")
hf_dataset = load_dataset("csv", data_files=data_files)

# Load tokenizer and prepare input text formatting
tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=True)
cols_to_merge = component_order.split("; ")
sep_token = f" {tokenizer.sep_token} " if tokenizer.sep_token else " "

# Preprocess dataset: format input text, rename target label and remove extra columns
preprocessed_ds = preprocess_dataset(hf_dataset, cols_to_merge, sep_token)

# Tokenize dataset
tokenized_ds = preprocessed_ds.map(
    lambda x: tokenizer(x["input_text"], truncation=True),
    batched=True,
    desc="Tokenizing input text"
)

# Itemize pos tagging 
pos_to_idx = {pos:i for i, pos in enumerate(pos_list)}
tokenized_ds = tokenized_ds.map(
    lambda x: {"pos_label": [pos_to_idx[val] for val in x['en_target_pos']]},
    batched=True,
    remove_columns=['en_target_pos', 'L1', 'input_text'],
    desc="Itemizing POS tagging"
)

Itemizing POS tagging: 100%|██████████| 2031/2031 [00:00<00:00, 135595.18 examples/s]


In [366]:
import math
sum(1 for x in tokenized_ds["train"]["freq_feature"] if math.isnan(x))
sum(1 for x in tokenized_ds["train"]["label"] if math.isnan(x))

0

In [906]:
backbone = 'xlm-roberta-base'
batch_size = 64
l1 = 'xx'
learning_rate_list1 = [2e-3,5e-3,2e-4,5e-4,2e-5,5e-5]
weight_decay_list1 = [0.1,0.01]
learning_rate_list2 = [2e-2, 5e-2, 2e-3, 5e-3]
weight_decay_list2 = [0, 0.1]
temp_list = [2, 20, 200]
dropout_list = [0.1, 0.2, 0.3]
token_agg_list = ['sentence', "mean_token"]
warmup_ratio = 0.1
epoch=5


temp = temp_list[0]
learning_rate1 = learning_rate_list1[0]
weight_decay1 = weight_decay_list1[0]
learning_rate2 = learning_rate_list2[0]
weight_decay2 = weight_decay_list2[0]
dropout = dropout_list[0]
token_agg = token_agg_list[0]

print(
    f"""temperature: {float(temp)}
        lr1: {float(learning_rate1)}
        weight_decay1: {float(weight_decay1)}
        lr2: {float(learning_rate2)}
        weight_decay2: {float(weight_decay2)}
        dropout: {float(dropout)}
        token_agg: {token_agg}
    """
)

def model_init_function():
        return CloseTrack_Predictor.from_pretrained(
            backbone, 
            num_labels=1,
            temp=temp,
            dropout=dropout,
            layer_wise=layer_wise,
            token_agg=token_agg
        )

try:
    logging.info(f"Fine-tuning model...")

    # Load dataset paths and Hugging Face DatasetDict
    data_files = load_data_paths(DATA_DIR, l1, "finetune")
    hf_dataset = load_dataset("csv", data_files=data_files)

    # Load tokenizer and prepare input text formatting
    tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=True)
    cols_to_merge = component_order.split("; ")
    sep_token = f" {tokenizer.sep_token} " if tokenizer.sep_token else " "

    # Preprocess dataset: format input text, rename target label and remove extra columns
    preprocessed_ds = preprocess_dataset(hf_dataset, cols_to_merge, sep_token)

    # Tokenize dataset
    tokenized_ds = preprocessed_ds.map(
        lambda x: tokenizer(x["input_text"], truncation=True),
        batched=True,
        desc="Tokenizing input text"
    )

    # Itemize pos tagging 
    pos_to_idx = {pos:i for i, pos in enumerate(pos_list)}
    tokenized_ds = tokenized_ds.map(
        lambda x: {"pos_label": [pos_to_idx[val] for val in x['en_target_pos']]},
        batched=True,
        remove_columns=['en_target_pos', 'L1', 'input_text'],
        desc="Itemizing POS tagging"
    )

    # Set up training arguments
    training_args = TrainingArguments(
        # output_dir=str(MODELS_DIR / model_name),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        # save_total_limit=1,
        # max_grad_norm=1.0, # gradient clipping
        num_train_epochs=int(epoch),
        per_device_train_batch_size=int(batch_size),
        per_device_eval_batch_size=int(batch_size),
        learning_rate=float(learning_rate1),
        weight_decay=float(weight_decay1),
        warmup_ratio=float(warmup_ratio),
        load_best_model_at_end=True,
        report_to="none",
        seed=1234,
        label_names=target_list
    )

    # Initialise trainer
    data_collator = DataCollatorWithPadding(tokenizer)
    # CustomDataCollator(tokenizer, custom_features=["freq_feature", "pos_label", "label"])

    # set customized optimizer
    # 1. Initialize the model first so we can access its parameters
    model = model_init_function()
    custom_param_names = ["numerator", "denominator"] # [] 
    
    custom_params = [p for n, p in model.named_parameters() if any(nd in n for nd in custom_param_names)]
    base_params = [p for n, p in model.named_parameters() if not any(nd in n for nd in custom_param_names)]

    # 3. Create Custom Optimizer with separate rules
    optimizer = AdamW([
        {
            "params": base_params, 
            "lr": training_args.learning_rate, 
            "weight_decay": training_args.weight_decay
        },
        {
            "params": custom_params, 
            "lr": learning_rate2,             # Massive LR for the scalars
            "weight_decay": weight_decay2     # STRICTLY ZERO weight decay
        }
    ])

    # 4. Rebuild the learning rate scheduler (Trainer usually does this automatically, 
    # but we must do it manually since we are overriding the optimizer)
    num_training_steps = (len(tokenized_ds["train"]) // training_args.per_device_train_batch_size) * training_args.num_train_epochs
    num_warmup_steps = int(num_training_steps * training_args.warmup_ratio)
    
    lr_scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    # Initialize the callback to watch your specific custom architecture weights
    debug_callback = OptimizerStateMonitor(
        params_to_watch=["scalar_weights", 'gamma'] 
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        data_collator=data_collator,
        optimizers=(optimizer, lr_scheduler),
        compute_metrics=compute_metrics,
        callbacks=[debug_callback]
    )
    
    trainer.train()
    
except Exception:
    logging.exception(f"Failed model")
    raise

temperature: 2.0
        lr1: 0.002
        weight_decay1: 0.1
        lr2: 0.02
        weight_decay2: 0.0
        dropout: 0.1
        token_agg: sentence
    


ERROR:root:Failed model
Traceback (most recent call last):
  File "/scratch/local/27881825/ipykernel_1849776/2740969936.py", line 101, in <module>
    model = model_init_function()
  File "/scratch/local/27881825/ipykernel_1849776/2740969936.py", line 35, in model_init_function
    return CloseTrack_Predictor.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        backbone,
        ^^^^^^^^^
    ...<4 lines>...
        token_agg=token_agg
        ^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/blue/jinnie.shin/zheli/.conda/envs/bea26/lib/python3.13/site-packages/transformers/modeling_utils.py", line 277, in _wrapper
    return func(*args, **kwargs)
  File "/blue/jinnie.shin/zheli/.conda/envs/bea26/lib/python3.13/site-packages/transformers/modeling_utils.py", line 4971, in from_pretrained
    model = cls(config, *model_args, **model_kwargs)
  File "/scratch/local/27881825/ipykernel_1849776/3345228232.py", line 61, in __init__
    self.token_agg == token_agg
    ^^^^^^^^^^^^^^

AttributeError: 'CloseTrack_Predictor' object has no attribute 'token_agg'